<a href="https://colab.research.google.com/github/CemKeremSahin/NIR-to-RGB-Colorization/blob/main/Notebooks/MUGAN_3Chanels_Train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
print(f"Sahip olduğun CPU çekirdek sayısı: {os.cpu_count()}")

In [ ]:
# --- 1. COLAB KURULUMLARI VE DRIVE BAĞLANTISI ---
import os

# Gerekli kütüphaneyi kuruyoruz
try:
    import segmentation_models_pytorch as smp
    print("Kütüphane zaten kurulu.")
except ImportError:
    print("Kütüphane kuruluyor...")
    os.system('pip install segmentation_models_pytorch')
    import segmentation_models_pytorch as smp

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Drive dan colab e aktarma
!cp -r /content/drive/MyDrive/4sensor_dataset/low_light_20230117/train /content/

In [ ]:
import os

# Görselinize göre güncellenmiş klasör yolları
DIR_785 = "/content/train/785nm"
DIR_850 = "/content/train/850nm"
DIR_940 = "/content/train/940nm"
RGB_FOLDER = "/content/train/gt_rgb"

print("--- 3 KANALLI (MULTI-BAND) VERİ SETİ KONTROLÜ BAŞLIYOR ---\n")

def veri_kontrol_3ch():
    klasorler = {
        "785nm": DIR_785,
        "850nm": DIR_850,
        "940nm": DIR_940,
        "GT_RGB": RGB_FOLDER
    }

    dosya_listeleri = {}

    # Tüm klasörlerin varlığını ve dosya sayılarını kontrol et
    for isim, yol in klasorler.items():
        if not os.path.exists(yol):
            print(f"HATA: {yol} klasörü bulunamadı! Lütfen yolu kontrol edin.")
            return

        # .bmp, .png veya .jpg uzantılı dosyaları okuyacak şekilde genişlettik
        dosyalar = sorted([f for f in os.listdir(yol) if f.lower().endswith(('.bmp', '.png', '.jpg'))])
        dosya_listeleri[isim] = dosyalar
        print(f"{isim} klasöründe {len(dosyalar)} adet görüntü dosyası bulundu.")

    # Tüm klasörlerdeki dosya sayılarının eşit olup olmadığını kontrol et
    uzunluklar = [len(liste) for liste in dosya_listeleri.values()]
    if len(set(uzunluklar)) == 1:
        print("\n>>> BAŞARILI: Tüm NIR bantları (785, 850, 940) ve RGB dosya sayıları EŞİT.")
    else:
        print("\n>>> UYARI: Dosya sayıları UYUŞMUYOR!")
        for isim, liste in dosya_listeleri.items():
            print(f"    {isim}: {len(liste)}")

    # Önceki kodunuzdaki 538 sayısı Real-World veri setinin toplamıdır[cite: 387].
    # Klasörünüzdeki sayı 458 çıkarsa bu normaldir çünkü 538 görüntünün 458'i eğitim (train), 80'i test içindir[cite: 388].

    # İlk 5 eşleşmeyi yan yana yazdırarak kontrol et
    print("\nÖrnek Eşleşmeler (İlk 5):")
    min_len = min(uzunluklar) if uzunluklar else 0

    if min_len > 0:
        for i in range(min(5, min_len)):
            f_785 = dosya_listeleri["785nm"][i]
            f_850 = dosya_listeleri["850nm"][i]
            f_940 = dosya_listeleri["940nm"][i]
            f_rgb = dosya_listeleri["GT_RGB"][i]
            print(f"  {f_785} | {f_850} | {f_940}  <--*-->  {f_rgb}")
    else:
        print("Klasörler boş olduğu için örnek gösterilemiyor.")

veri_kontrol_3ch()

--- 3 KANALLI (MULTI-BAND) VERİ SETİ KONTROLÜ BAŞLIYOR ---

785nm klasöründe 206 adet görüntü dosyası bulundu.
850nm klasöründe 206 adet görüntü dosyası bulundu.
940nm klasöründe 206 adet görüntü dosyası bulundu.
GT_RGB klasöründe 206 adet görüntü dosyası bulundu.

>>> BAŞARILI: Tüm NIR bantları (785, 850, 940) ve RGB dosya sayıları EŞİT.

Örnek Eşleşmeler (İlk 5):
  001.bmp | 001.bmp | 001.bmp  <--*-->  0001.bmp
  002.bmp | 002.bmp | 002.bmp  <--*-->  0002.bmp
  003.bmp | 003.bmp | 003.bmp  <--*-->  0003.bmp
  004.bmp | 004.bmp | 004.bmp  <--*-->  0004.bmp
  005.bmp | 005.bmp | 005.bmp  <--*-->  0005.bmp


In [ ]:
import os
import cv2
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import Normalize
import torchvision.models as models
from tqdm import tqdm
from google.colab import runtime, drive # Gerekli importlar eklendi

# --- 1. (TF32 AKTİF VE AMP İÇİN PERFORMANS AYARLARI) ---
# A100 ve modern NVIDIA GPU'lar için TF32'yi etkinleştirir, standart fp32 hassasiyetini 'high' yapar.
torch.backends.cudnn.allow_tf32 = True
torch.backends.cuda.matmul.allow_tf32 = True
#torch.backends.cuda.matmul.fp32_precision = 'high'
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Kullanılan Cihaz: {DEVICE}")

# --- 2. AYARLAR VE YOLLAR ---
ROOT_PATH = "/content/train"
# Lütfen Drive'ı önceden bağladığınızdan emin olun.
SAVE_PATH = "/content/drive/MyDrive/Computer_Vision/Model_Weights/MUGAN_3Channels"
os.makedirs(SAVE_PATH, exist_ok=True)

PATCH_SIZE = 256
BATCH_SIZE = 16
NUM_EPOCHS = 3500
LEARNING_RATE = 2e-4
SAVE_EVERY_N_EPOCHS = 500

# --- 3. MUGAN MIMARISI (GENERATOR) ---
class MixedSkipBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(MixedSkipBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class MUGANGenerator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3): # 3 Kanal Giriş ve Çıkış
        super(MUGANGenerator, self).__init__()
        self.enc1 = MixedSkipBlock(in_channels, 64)
        self.enc2 = MixedSkipBlock(64, 128)
        self.enc3 = MixedSkipBlock(128, 256)
        self.enc4 = MixedSkipBlock(256, 512)
        self.pool = nn.MaxPool2d(2)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.bridge = MixedSkipBlock(512, 1024)
        self.dec4 = MixedSkipBlock(1024 + 512, 512)
        self.dec3 = MixedSkipBlock(512 + 256, 256)
        self.dec2 = MixedSkipBlock(256 + 128, 128)
        self.dec1 = MixedSkipBlock(128 + 64, 64)
        self.final = nn.Sequential(nn.Conv2d(64, out_channels, kernel_size=1), nn.Tanh())

    def forward(self, x):
        s1 = self.enc1(x); p1 = self.pool(s1)
        s2 = self.enc2(p1); p2 = self.pool(s2)
        s3 = self.enc3(p2); p3 = self.pool(s3)
        s4 = self.enc4(p3); p4 = self.pool(s4)
        b = self.bridge(p4)
        d4 = self.dec4(torch.cat([self.upsample(b), s4], dim=1))
        d3 = self.dec3(torch.cat([self.upsample(d4), s3], dim=1))
        d2 = self.dec2(torch.cat([self.upsample(d3), s2], dim=1))
        d1 = self.dec1(torch.cat([self.upsample(d2), s1], dim=1))
        return self.final(d1)

# --- 4. DISCRIMINATOR (PATCHGAN) ---
class PatchGAN(nn.Module):
    def __init__(self):
        super().__init__()
        def block(in_f, out_f, stride=2, norm=True):
            layers = [nn.Conv2d(in_f, out_f, 4, stride, 1)]
            if norm: layers.append(nn.InstanceNorm2d(out_f))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return nn.Sequential(*layers)

        self.model = nn.Sequential(
            # IR (3 kanal) + RGB (3 kanal) = 6 Kanal Giriş
            block(6, 64, norm=False),
            block(64, 128), block(128, 256), block(256, 512, stride=1),
            nn.Conv2d(512, 1, 4, 1, 1)
        )
    def forward(self, ir, rgb): return self.model(torch.cat((ir, rgb), 1))

# --- 5. DATASET VE LOSS ---
class MultiBandNIRDatasetRAM(Dataset):
    def __init__(self, root_path, patch_size=256):
        self.patch_size = patch_size
        self.data_cache = []

        dir_785 = os.path.join(root_path, "785nm")
        dir_850 = os.path.join(root_path, "850nm")
        dir_940 = os.path.join(root_path, "940nm")
        rgb_dir = os.path.join(root_path, "gt_rgb") # Klasör adı ilk koddaki gibi gt_rgb yapıldı

        files_785 = sorted([f for f in os.listdir(dir_785) if f.lower().endswith(('.bmp', '.png', '.jpg'))])
        files_850 = sorted([f for f in os.listdir(dir_850) if f.lower().endswith(('.bmp', '.png', '.jpg'))])
        files_940 = sorted([f for f in os.listdir(dir_940) if f.lower().endswith(('.bmp', '.png', '.jpg'))])
        rgb_files = sorted([f for f in os.listdir(rgb_dir) if f.lower().endswith(('.bmp', '.png', '.jpg'))])

        print(f"📥 3 Kanallı Veriler RAM'e yükleniyor...")
        for f7, f8, f9, fr in tqdm(zip(files_785, files_850, files_940, rgb_files), total=len(files_785)):
            img_785 = cv2.imread(os.path.join(dir_785, f7), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
            img_850 = cv2.imread(os.path.join(dir_850, f8), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
            img_940 = cv2.imread(os.path.join(dir_940, f9), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0

            # 3 bandı kanal boyutunda birleştir (H, W, 3)
            ir_img = np.stack([img_785, img_850, img_940], axis=-1)

            rgb_img = cv2.imread(os.path.join(rgb_dir, fr), cv2.IMREAD_COLOR)
            rgb_img = cv2.cvtColor(rgb_img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

            self.data_cache.append({'ir': ir_img, 'rgb': rgb_img})

    def __len__(self): return len(self.data_cache)
    def __getitem__(self, idx):
        d = self.data_cache[idx]
        ir = torch.from_numpy(d['ir']).permute(2, 0, 1) * 2.0 - 1.0
        rgb = torch.from_numpy(d['rgb']).permute(2, 0, 1) * 2.0 - 1.0
        i, j, h, w = transforms.RandomCrop.get_params(ir, (self.patch_size, self.patch_size))
        return transforms.functional.crop(ir, i, j, h, w), transforms.functional.crop(rgb, i, j, h, w)

class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.vgg = models.vgg16(weights='DEFAULT').features[:16].eval().to(DEVICE)
        for p in self.vgg.parameters(): p.requires_grad = False
        self.criterion = nn.L1Loss()
        self.normalize = Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # VGG için norm eklendi

    def forward(self, fake, real):
        norm_fake = self.normalize((fake + 1.0) / 2.0)
        norm_real = self.normalize((real + 1.0) / 2.0)
        return self.criterion(self.vgg(norm_fake), self.vgg(norm_real))

# --- 6. AMP İLE EĞİTİM KURULUMU VE DÖNGÜSÜ ---
model_G = MUGANGenerator(in_channels=3, out_channels=3).to(DEVICE) # 3 Kanal Giriş/Çıkış
model_D = PatchGAN().to(DEVICE)

optimizer_G = optim.Adam(model_G.parameters(), lr=LEARNING_RATE, betas=(0.5, 0.999))
optimizer_D = optim.Adam(model_D.parameters(), lr=LEARNING_RATE, betas=(0.5, 0.999))

criterion_GAN = nn.BCEWithLogitsLoss()
criterion_MAE = nn.L1Loss()
criterion_percep = PerceptualLoss()

loader = DataLoader(MultiBandNIRDatasetRAM(ROOT_PATH, PATCH_SIZE), BATCH_SIZE, shuffle=True, pin_memory=True)

# --- (GradScaler Başlatılıyor - 'cuda' cihazı için) ---
scaler = torch.amp.GradScaler('cuda')

print(f"\n🚀 3 Kanallı MUGAN Eğitimi A100 Üzerinde AMP ve GradScaler ile Başlıyor...")

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_g, epoch_d = 0, 0
    loop = tqdm(loader, leave=False)
    for ir, real in loop:
        ir, real = ir.to(DEVICE), real.to(DEVICE)

        # ============================================
        #           DISCRIMINATOR EĞİTİMİ
        # ============================================
        optimizer_D.zero_grad()

        # Forward pass için autocast context'i aç (fp16 devreye girer)
        with torch.amp.autocast('cuda', dtype=torch.float16):
            # Gerçek Görüntü
            real_pred = model_D(ir, real)
            l_d_real = criterion_GAN(real_pred, torch.ones_like(real_pred))

            # Sahte Görüntü (Generator'dan gradyan akışını kesmek için detach yapıyoruz)
            fake = model_G(ir)
            fake_pred = model_D(ir, fake.detach())
            l_d_fake = criterion_GAN(fake_pred, torch.zeros_like(fake_pred))

            # Toplam Discriminator Kaybı
            l_d = (l_d_real + l_d_fake) * 0.5

        # --- (Loss'u ölçekle ve backward yap) ---
        scaler.scale(l_d).backward()
        # --- (Optimizer step'ini scaler ile yap) ---
        scaler.step(optimizer_D)
        scaler.update() # Scaler'ı bir sonraki iterasyon için güncelle


        # ============================================
        #           GENERATOR EĞİTİMİ
        # ============================================
        optimizer_G.zero_grad()

        # Generator forward pass için de autocast'i açıyoruz
        with torch.amp.autocast('cuda', dtype=torch.float16):
            fake = model_G(ir) # Fake görüntüyü tekrar oluşturuyoruz (gradyanları tutmak için)
            fake_pred = model_D(ir, fake) # Detach yok! Gradyanlar Generator'a akacak

            # Kayıplar (Ağırlıkları: MAE x50, Perceptual x10)
            l_g_mae = criterion_MAE(fake, real) * 50.0
            l_g_percep = criterion_percep(fake, real) * 10.0
            l_g_gan = criterion_GAN(fake_pred, torch.ones_like(fake_pred))

            # Toplam Generator Kaybı
            l_g = l_g_gan + l_g_mae + l_g_percep

        # --- (Loss'u ölçekle ve backward yap) ---
        scaler.scale(l_g).backward()
        # --- (Optimizer step'ini scaler ile yap) ---
        scaler.step(optimizer_G)
        scaler.update() # Scaler'ı güncelle


        # İstatistik güncelleme (Yazdırırken ölçeklenmemiş orjinal loss değerlerini alıyoruz)
        epoch_g += l_g.item(); epoch_d += l_d.item()
        loop.set_description(f"Epoch [{epoch}/{NUM_EPOCHS}]")
        loop.set_postfix(G=l_g.item(), D=l_d.item())

    if epoch % 100 == 0:
        print(f"Epoch {epoch} | G_Loss: {epoch_g/len(loader):.4f} | D_Loss: {epoch_d/len(loader):.4f}")

    if epoch % SAVE_EVERY_N_EPOCHS == 0:
      torch.save({
        'epoch': epoch,
        'model_G_state_dict': model_G.state_dict(),
        'model_D_state_dict': model_D.state_dict(),
        'optimizer_G_state_dict': optimizer_G.state_dict(),
        'optimizer_D_state_dict': optimizer_D.state_dict(),
        'scaler_state_dict': scaler.state_dict(), # Kararlılık için scaler state'ini de kaydetmeliyiz
      }, os.path.join(SAVE_PATH, f"MUGAN_3channels_Train_epoch_{epoch}.pth"))

print("✅ Eğitim bitti. Drive senkronize ediliyor...")
drive.flush_and_unmount() # Drive dosyalarını senkronize et ve bağlantıyı kes
print("✅ Senkronizasyon başarılı. Oturum otomatik olarak kapatılıyor...")
runtime.unassign() # Colab oturumunu kapat